[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# SQLModel in FastAPI


## What you will be able to do

Put these models behind HTTP: a session that belongs to one request and closes with it, a create
model as the body a route accepts, a public model as what it answers with, and the status codes for
a hero that is not there and a name that is taken. Load a relationship the response will need, and
recognize what happens when you do not. Write the test setup the **A Small Service** notebook
reuses, with an engine in memory that the application's thread can actually see. Recognize the
failures: two response models that refer to each other, a relationship read after its session
closed, and the two that come from an in-memory database and another thread.


## The idea

### The problem

Everything so far has been one program talking to one database. A service is not that. It answers
many requests at once, each of which has to have a session of its own and give it back when it is
finished; it takes data from strangers, which has to be checked before it reaches a model; and it
answers with JSON, which has to contain what a client may see and nothing else.

FastAPI and SQLModel are written by the same author and fit together without glue, which makes it
easy to write a route that works and is wrong in a way nobody sees until later. Returning a table
model publishes the columns nobody meant to publish. Taking a table model as the body skips every
check, because a table model's constructor does not validate. Letting a response model reach a
relationship that was never loaded fails only once the session has closed, which is after the route
returned, in code that looks blameless. And two response models that point at each other make a
response that cannot be built at all.

Every one of those is an earlier notebook arriving at HTTP, with a short answer that follows it.

### What the pieces are

> **`Depends(get_session)`** gives a route a session from a generator that yields one and closes it
> when the request is finished, so every request has its own and none of them outlive it. The body a
> route takes is a model **with no table**, so it is validated and a bad field becomes a **422**
> naming it. **`response_model`** says what the answer is built from, which is a public model, and
> FastAPI copies only the fields that model declares. **`HTTPException(status_code=404)`** is what a
> missing row becomes, and a caught **`IntegrityError`** is what a repeated name becomes, as a
> **409**. For tests, **`app.dependency_overrides[get_session]`** swaps the session for one on
> another engine.

### Why it works that way

- **A session belongs to a request.** It holds a connection and a transaction, and a request is the
  unit of work that both are for.
- **The body is checked before it is a row.** A create model with no table validates in its
  constructor, and FastAPI turns what it refuses into a 422 that names the field.
- **A response model is a filter, not a suggestion.** FastAPI builds the answer out of the fields
  the response model declares, so a secret column cannot reach a client by accident.
- **The response is built after the route returns.** Anything it reads has to be loaded before the
  session closes, which is what the loading options in the **Loading and N+1** notebook are for.
- **Nesting has to stop somewhere.** A hero whose team has heroes whose team has heroes is a loop,
  and Pydantic detects it rather than following it.

### Where this shows up

Every service built on these two libraries. The **APIs and JSON** guide is where FastAPI itself is
taught, including status codes, request bodies and the documentation it generates; this notebook
assumes that and is about the seam between a route and a session. The **A Small Service** notebook
puts the whole of it together with a migration and a test suite.

### What this notebook covers

- A session that belongs to one request
- What a route takes, and what it answers with
- A table model as the body, which checks nothing
- The hero that is not there, and the name that is taken
- A relationship in the answer, loaded on purpose
- A test client, and an engine two threads can share
- A small API, finished
- Four failures, from two models that refer to each other to a table the application's thread cannot
  see

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from fastapi import Depends, FastAPI
from fastapi.testclient import TestClient
from sqlmodel import Field, Session, SQLModel, create_engine, select


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    secret_name: str


class HeroPublic(SQLModel):
    id: int
    name: str


engine = create_engine("sqlite:///heroes.db")   # a file: the client runs the app in another thread
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    session.add(Hero(name="Deadpond", secret_name="Dive Wilson"))
    session.commit()


def get_session():
    with Session(engine) as session:
        yield session


app = FastAPI()


@app.get("/heroes", response_model=list[HeroPublic])
def read_heroes(session: Session = Depends(get_session)):
    return session.exec(select(Hero)).all()


print(TestClient(app).get("/heroes").json())
```

```
[{'id': 1, 'name': 'Deadpond'}]
```

The route returned `Hero` objects, and the answer has no secret name in it: `response_model` built
each item out of the fields `HeroPublic` declares and ignored the rest. The session came from
`Depends`, which opened one for that request and closed it afterwards.


## Setup

Sixteen imports, three packages installed where they are missing, the cast, two helpers, six
classes, the engine, the session dependency, and the database built and loaded.

- `sqlmodel` is the library, and `SQLModel`, `Field`, `Relationship`, `Session`, `create_engine` and
  `select`, from it, are the classes, the session and the reading. Colab has neither SQLModel nor
  httpx pinned as this notebook wants them, so the cell installs what is missing and `version` and
  `PackageNotFoundError`, from `importlib.metadata`, find out what that is
- `FastAPI`, `Depends` and `HTTPException`, from `fastapi`, are the application, the session a route
  asks for and the answers that are not 200, and `ResponseValidationError`, from
  `fastapi.exceptions`, is what a response that cannot be built raises
- `TestClient`, from `fastapi.testclient`, sends requests to an application in this process, with no
  server and no port. The import is wrapped because starlette warns about its HTTP client on some
  installations, and a warning carries the path of a file inside the library
- `event`, `insert`, `IntegrityError`, `selectinload` and `StaticPool` are the pragma on every
  connection, the rows `build` loads, the refusal a repeated name raises, the loading option from the
  **Loading and N+1** notebook, and the pool that lets two threads share one database in memory
- `re` takes memory addresses out of a message, `contextlib` and `warnings` quiet the import above,
  `Path` names the database file, and `shutil` removes the scratch folder at the start and at the end
- `TEAMS` and `HEROES` are the cast, which `build` loads

The classes are the guide's, with the family from the **Create, Read and Update Models** notebook:
`HeroCreate` for what a client sends, `HeroPublic` for what it may see, `TeamPublic`, and
`HeroWithTeam`, which is one level deeper and stops there. `get_session` is the whole of the seam
between FastAPI and SQLModel, and it is four lines.


In [1]:
import contextlib
import re
import shutil
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

for package, pin in (("sqlmodel", "sqlmodel==0.0.42"), ("fastapi", "fastapi==0.141.1"), ("httpx", "httpx==0.28.1")):
    try:
        version(package)
    except PackageNotFoundError:                                    # install what this runtime is missing
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore", pin],
                       check=True)

import sqlmodel
from fastapi import Depends, FastAPI, HTTPException
from fastapi.exceptions import ResponseValidationError
from sqlalchemy import event, insert
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import selectinload
from sqlalchemy.pool import StaticPool
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select

with warnings.catch_warnings():                                     # starlette warns here on some installations
    warnings.simplefilter("ignore")
    from fastapi.testclient import TestClient

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, unique=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")


class HeroCreate(SQLModel):                                         # what a client may send
    name: str = Field(max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = None
    team_id: int | None = None


class HeroPublic(SQLModel):                                         # what a client may see
    id: int
    name: str
    age: int | None = None


class TeamPublic(SQLModel):
    id: int
    name: str
    headquarters: str


class HeroWithTeam(HeroPublic):                                     # one level deeper, and no further
    team: TeamPublic | None = None


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

def get_session():
    """One session for one request, closed when the request is finished."""
    with Session(engine) as session:
        yield session

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

print("sqlmodel", sqlmodel.__version__, "| fastapi", version("fastapi"), "| httpx", version("httpx"))


sqlmodel 0.0.42 | fastapi 0.141.1 | httpx 0.28.1


## Worked examples

### A session that belongs to one request

`Depends(get_session)` is how a route asks for one. The generator opens a session, yields it, and
closes it when the request is finished:


In [2]:
heroes_api = FastAPI()


@heroes_api.get("/heroes", response_model=list[HeroPublic])
def read_heroes(session: Session = Depends(get_session)):
    return session.exec(select(Hero).order_by(Hero.name).limit(3)).all()


client = TestClient(heroes_api)
print(client.get("/heroes").status_code, client.get("/heroes").json())


200 [{'id': 5, 'name': 'Black Lion', 'age': 35}, {'id': 7, 'name': 'Captain North America', 'age': 93}, {'id': 1, 'name': 'Deadpond', 'age': None}]


Three heroes, each with an id, a name and an age, and no secret name anywhere: the route returned
`Hero` objects and `response_model` built the answer out of `HeroPublic`. `TestClient` sends the
request in this process, with no server and no port, which is also how the tests in the
**A Small Service** notebook work.

### What a route takes, and what it answers with

The body is a model with no table, so it is checked before anything reaches a row:


In [3]:
@heroes_api.post("/heroes", response_model=HeroPublic, status_code=201)
def make_hero(arriving: HeroCreate, session: Session = Depends(get_session)):
    hero = Hero.model_validate(arriving)
    session.add(hero)
    session.commit()
    session.refresh(hero)
    return hero


made = client.post("/heroes", json={"name": "Ghost Girl", "secret_name": "Ana Vega", "age": 27})
print("created:", made.status_code, made.json())

refused = client.post("/heroes", json={"name": "Mystery Man", "secret_name": "Unknown", "age": "old"})
print("refused:", refused.status_code, refused.json()["detail"][0]["loc"],
      refused.json()["detail"][0]["msg"])


created: 201 {'id': 9, 'name': 'Ghost Girl', 'age': 27}
refused: 422 ['body', 'age'] Input should be a valid integer, unable to parse string as an integer


A 201 with the public fields, and a 422 that names the field and what was wrong with it, in the
shape every FastAPI client already knows how to read. Nothing in the route wrote out a field list:
`HeroCreate` says what may arrive, `HeroPublic` says what may leave, and
`Hero.model_validate(arriving)` is the one line between them.

### A table model as the body, which checks nothing

The same route with `Hero` as the body looks tidier and is a hole:


In [4]:
loose_api = FastAPI()


@loose_api.post("/heroes", response_model=HeroPublic, status_code=201)
def make_hero_loosely(hero: Hero, session: Session = Depends(get_session)):
    session.add(hero)
    session.commit()
    session.refresh(hero)
    return hero


loose = TestClient(loose_api)
try:
    loose.post("/heroes", json={"name": "Loose Cannon", "secret_name": "Unknown", "age": "old"})
except ResponseValidationError as error:
    print(type(error).__name__ + ":", str(error).splitlines()[1].strip().split(", 'input'")[0] + "}")


ResponseValidationError: {'type': 'int_parsing', 'loc': ('response', 'age'), 'msg': 'Input should be a valid integer, unable to parse string as an integer'}


The body was not checked. A table model's constructor validates nothing, which the
**Validation and table=True** notebook showed, and FastAPI builds the body with it, so the word
`old` went into a field annotated `int | None` and straight into the row. The failure came later,
when the answer was built out of `HeroPublic`, which does check, and the client got a 500 for a
mistake it made in its own request.

A create model is the fix, and the reason it is worth the extra class: with `HeroCreate` the same
request is a 422 that names the field.

### The hero that is not there, and the name that is taken

Two answers a service owes: 404 for a row that does not exist, and 409 for one that already does:


In [5]:
@heroes_api.get("/heroes/{hero_id}", response_model=HeroPublic)
def read_hero(hero_id: int, session: Session = Depends(get_session)):
    hero = session.get(Hero, hero_id)
    if hero is None:
        raise HTTPException(status_code=404, detail="no hero with that id")
    return hero


@heroes_api.post("/heroes/unique", response_model=HeroPublic, status_code=201)
def make_unique_hero(arriving: HeroCreate, session: Session = Depends(get_session)):
    session.add(Hero.model_validate(arriving))
    try:
        session.commit()
    except IntegrityError:
        session.rollback()                                          # the session is usable again
        raise HTTPException(status_code=409, detail=f"{arriving.name} is taken")
    return session.exec(select(Hero).where(Hero.name == arriving.name)).one()


print("found    :", client.get("/heroes/1").status_code, client.get("/heroes/1").json())
print("not found:", client.get("/heroes/999").status_code, client.get("/heroes/999").json())
taken = client.post("/heroes/unique", json={"name": "Deadpond", "secret_name": "Someone Else"})
print("taken    :", taken.status_code, taken.json())


found    : 200 {'id': 1, 'name': 'Deadpond', 'age': None}
not found: 404 {'detail': 'no hero with that id'}
taken    : 409 {'detail': 'Deadpond is taken'}


`session.get` returns `None` rather than raising, as the **Sessions** notebook showed, so the route
is what turns that into a 404. The 409 is the database's refusal caught and translated: without the
`except`, the `IntegrityError` would reach FastAPI as an unhandled exception and the client would
get a 500 for something it could have fixed. The `rollback` is what the **Sessions** notebook
insisted on, and it matters more here, because the session is about to be given back.

### A relationship in the answer, loaded on purpose

A response model that reaches a relationship needs that relationship loaded before the session
closes, and the session closes after the response is built:


In [6]:
@heroes_api.get("/heroes/{hero_id}/with-team", response_model=HeroWithTeam)
def read_hero_with_team(hero_id: int, session: Session = Depends(get_session)):
    hero = session.exec(select(Hero).where(Hero.id == hero_id)
                        .options(selectinload(Hero.team))).one_or_none()
    if hero is None:
        raise HTTPException(status_code=404, detail="no hero with that id")
    return hero


answer = client.get("/heroes/1/with-team")
print(answer.status_code, answer.json())


200 {'id': 1, 'name': 'Deadpond', 'age': None, 'team': {'id': 2, 'name': 'Z-Force', 'headquarters': "Sister Margaret's Bar"}}


One request, two statements, and a hero with a team inside it. `selectinload` is the option from the
**Loading and N+1** notebook, and in a route it does two jobs at once: it keeps the request from
sending one query per hero, and it makes sure the team is in memory when the answer is built. The
second of the Common errors is what happens when the session has closed by then.

### A test client, and an engine two threads can share

A test wants a database of its own, empty at the start and gone at the end. An engine in memory is
the obvious choice and does not work on the first try:


In [7]:
def try_engine(label, test_engine):
    """Point the application at another engine, ask it for heroes, and say what happened."""
    SQLModel.metadata.create_all(test_engine)
    with Session(test_engine) as session:
        session.add(Hero(name="Test Hero", secret_name="Someone"))
        session.commit()

    def override():
        with Session(test_engine) as session:
            yield session

    heroes_api.dependency_overrides[get_session] = override
    try:
        answered = TestClient(heroes_api).get("/heroes")
        print(f"{label}: {answered.status_code} {answered.json()}")
    except Exception as error:
        print(f"{label}: {type(error).__name__}: {str(error).splitlines()[0].split('. The object')[0]}")
    finally:
        heroes_api.dependency_overrides.clear()                     # and the engine is left for Python



try_engine("in memory      ", create_engine("sqlite://"))
try_engine("with StaticPool", create_engine("sqlite://", poolclass=StaticPool))
try_engine("and no guard   ", create_engine("sqlite://", poolclass=StaticPool,
                                            connect_args={"check_same_thread": False}))


in memory      : OperationalError: (sqlite3.OperationalError) no such table: hero
with StaticPool: ProgrammingError: (sqlite3.ProgrammingError) SQLite objects created in a thread can only be used in that same thread
and no guard   : 200 [{'id': 1, 'name': 'Test Hero', 'age': None}]


Three engines and three outcomes, and the two arguments are now explained rather than copied. A
plain in-memory engine gives every thread a connection of its own, and a connection of its own is a
database of its own, so the tables the test made are not there for the request's thread.
`StaticPool` keeps one connection and hands the same one to everybody, which runs into `sqlite3`'s
rule that a connection belongs to the thread that made it. `check_same_thread=False` turns that rule
off, which is safe here because the pool hands the connection to one thread at a time.

`dependency_overrides` is the other half: the application is unchanged, and the session it is given
comes from somewhere else for the length of the test.

### A small API, finished

The pieces of this notebook in one application, with a test that proves it:


In [8]:
api = FastAPI()


@api.post("/heroes", response_model=HeroPublic, status_code=201)
def create_hero(arriving: HeroCreate, session: Session = Depends(get_session)):
    hero = Hero.model_validate(arriving)
    session.add(hero)
    try:
        session.commit()
    except IntegrityError:
        session.rollback()
        raise HTTPException(status_code=409, detail=f"{arriving.name} is taken")
    session.refresh(hero)
    return hero


@api.get("/heroes/{hero_id}", response_model=HeroWithTeam)
def show_hero(hero_id: int, session: Session = Depends(get_session)):
    hero = session.exec(select(Hero).where(Hero.id == hero_id)
                        .options(selectinload(Hero.team))).one_or_none()
    if hero is None:
        raise HTTPException(status_code=404, detail="no hero with that id")
    return hero


def client_on_a_fresh_database():
    """A TestClient for the api, on an empty database this process shares with it."""
    test_engine = create_engine("sqlite://", poolclass=StaticPool,
                                connect_args={"check_same_thread": False})
    SQLModel.metadata.create_all(test_engine)

    def override():
        with Session(test_engine) as session:
            yield session

    api.dependency_overrides[get_session] = override
    return TestClient(api), test_engine


testing, test_engine = client_on_a_fresh_database()
print("created  :", testing.post("/heroes", json={"name": "Deadpond", "secret_name": "Dive Wilson"}).json())
print("read back:", testing.get("/heroes/1").json())
print("taken    :", testing.post("/heroes", json={"name": "Deadpond", "secret_name": "Other"}).status_code)
print("missing  :", testing.get("/heroes/99").status_code)
print("refused  :", testing.post("/heroes", json={"name": "X"}).status_code)

api.dependency_overrides.clear()


created  : {'id': 1, 'name': 'Deadpond', 'age': None}
read back: {'id': 1, 'name': 'Deadpond', 'age': None, 'team': None}
taken    : 409
missing  : 404
refused  : 422


Five requests and five answers: a hero created, the same hero read back with the team it does not
have, a repeated name refused with 409, a missing hero with 404, and a body with no secret name
refused with 422 before any database was asked. Every one of those came from a piece of an earlier
notebook.

### Where each part came from

| In the small API | What it relies on | The notebook it came from |
|---|---|---|
| `Depends(get_session)` | a session opened and closed around one unit of work | the **Sessions** notebook |
| `HeroCreate` as the body | a model with no table, whose constructor validates | the **Validation and table=True** notebook |
| `response_model=HeroPublic` | a model carrying only what may be published | the **Create, Read and Update Models** notebook |
| `selectinload(Hero.team)` | a relationship loaded before the session closes | the **Loading and N+1** notebook |
| `except IntegrityError` and `rollback` | a refusal caught, and a session left usable | the **Sessions** notebook |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/13-sqlmodel-in-fastapi-solutions.ipynb).

**1.** Add a route that lists teams as `TeamPublic`, and call it.


In [9]:
# your code here


**2.** Add a route that creates a team from a model with no table, answering 201, and show what it
does with a body that has no headquarters.


In [10]:
# your code here


**3.** Add a route for one team that answers 404 when there is no such team, and call it twice.


In [11]:
# your code here


**4.** Write `TeamWithHeroes`, a response model carrying a team and its heroes as `HeroPublic`, and
a route that loads the heroes on purpose.


In [12]:
# your code here


**5.** Add a route that deletes a hero, answering 204 with no body, and 404 when there is nothing to
delete.


In [13]:
# your code here


**6.** Write a function that gives you a `TestClient` on a fresh in-memory database, use it to
create two heroes and list them, and clear the override afterwards.


In [14]:
# your code here


## Common errors

### fastapi.exceptions.ResponseValidationError: {'type': 'recursion_loop', 'msg': 'Recursion error - cyclic reference detected'}


In [15]:
class TeamWithHeroes(SQLModel):
    id: int
    name: str
    heroes: list["HeroInTeam"] = []


class HeroInTeam(SQLModel):
    id: int
    name: str
    team: TeamWithHeroes | None = None                              # and back again


TeamWithHeroes.model_rebuild()
loop_api = FastAPI()


@loop_api.get("/teams/{team_id}", response_model=TeamWithHeroes)
def read_team(team_id: int, session: Session = Depends(get_session)):
    return session.get(Team, team_id)


try:
    TestClient(loop_api).get("/teams/1")
except ResponseValidationError as error:                            # its 'input' is an object, printed in no fixed order
    print(type(error).__name__ + ":", str(error).splitlines()[1].strip().split(", 'input'")[0] + "}")


ResponseValidationError: {'type': 'recursion_loop', 'loc': ('response', 'heroes', 0, 'team', 'heroes', 0), 'msg': 'Recursion error - cyclic reference detected'}


A team carries its heroes, each hero carries its team, and that team carries its heroes. Pydantic
walks it, notices it is walking in a circle, and refuses rather than following it until the stack
runs out. The `loc` in the message is the useful part: it names the path through the models where
the loop closed, which is where one of the two has to stop.

The answer is to have a version of each model that does not nest further, which is what
`HeroPublic` and `TeamPublic` are for in this notebook: `HeroWithTeam` carries a `TeamPublic`, and a
`TeamPublic` carries no heroes at all.

### fastapi.exceptions.ResponseValidationError: {'type': 'get_attribute_error', 'msg': "Error extracting attribute: DetachedInstanceError"}


In [16]:
closed_api = FastAPI()


@closed_api.get("/heroes/{hero_id}", response_model=HeroWithTeam)
def read_hero_from_a_closed_session(hero_id: int):
    with Session(engine) as session:                                # a session of the route's own
        return session.get(Hero, hero_id)                           # closed before the answer is built


try:
    TestClient(closed_api).get("/heroes/1")
except ResponseValidationError as error:
    print(type(error).__name__ + ":", message(error).splitlines()[1].strip().split(", 'input'")[0] + "}")


ResponseValidationError: {'type': 'get_attribute_error', 'loc': ('response', 'team'), 'msg': "Error extracting attribute: DetachedInstanceError: Parent instance <Hero at 0x...> is not bound to a Session; lazy load operation of attribute 'team' cannot proceed (Background on this error at: https://sqlalche.me/e/20/bhk3)"}


The route returned before the answer was built, and its `with` block closed the session on the way
out. FastAPI then asked the hero for its team, which was never loaded, and there was no session left
to load it with. The message wraps the `DetachedInstanceError` from the **Relationships** notebook
inside a response error, which is what makes it hard to place the first time.

Both fixes are in this notebook already: take the session from `Depends(get_session)`, which stays
open until the response is finished, and load what the response model will read, with
`selectinload`.

### sqlalchemy.exc.OperationalError: (sqlite3.OperationalError) no such table: hero


In [17]:
naive = create_engine("sqlite://")                                  # a database in memory, and no pool named
SQLModel.metadata.create_all(naive)
with Session(naive) as session:
    session.add(Hero(name="Test Hero", secret_name="Someone"))
    session.commit()
    print("the test can see:", len(session.exec(select(Hero)).all()), "hero")


def override():
    with Session(naive) as session:
        yield session


heroes_api.dependency_overrides[get_session] = override
try:
    TestClient(heroes_api).get("/heroes")
finally:
    heroes_api.dependency_overrides.clear()


the test can see: 1 hero


OperationalError: (sqlite3.OperationalError) no such table: hero
[SQL: SELECT hero.id, hero.name, hero.secret_name, hero.age, hero.team_id 
FROM hero ORDER BY hero.name
 LIMIT ? OFFSET ?]
[parameters: (3, 0)]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

The test made a table and wrote a row, and the request found no table at all. A database in memory
belongs to the connection that made it, and SQLAlchemy's default pool for one gives every thread its
own connection, so the thread `TestClient` runs the application in got an empty database with no
tables in it.

`poolclass=StaticPool` is what makes every thread share one connection, and therefore one database,
which the worked example above measured.

### sqlalchemy.exc.ProgrammingError: (sqlite3.ProgrammingError) SQLite objects created in a thread can only be used in that same thread.


In [18]:
shared = create_engine("sqlite://", poolclass=StaticPool)           # one connection, and sqlite3's rule
SQLModel.metadata.create_all(shared)


def override_shared():
    with Session(shared) as session:
        yield session


heroes_api.dependency_overrides[get_session] = override_shared
try:
    TestClient(heroes_api).get("/heroes")
except Exception as error:
    print(type(error).__name__ + ":", str(error).splitlines()[0].split(". The object")[0])
finally:
    heroes_api.dependency_overrides.clear()


ProgrammingError: (sqlite3.ProgrammingError) SQLite objects created in a thread can only be used in that same thread


The pool now hands the same connection to whoever asks, and `sqlite3` refuses to let a connection
made in one thread be used in another. The message names both threads, which is why only its first
sentence is printed here.

`connect_args={"check_same_thread": False}` turns that check off. It is safe with `StaticPool`
because the pool gives the connection to one thread at a time, and it is not a setting to copy into
an application's engine, where a real pool and real concurrency make the check worth having.

None of the engines in this notebook's tests is disposed. `dispose()` closes every connection from
whichever thread calls it, and `sqlite3` refuses to close one that was made in another, which the
pool then writes to the screen; a database in memory goes when nothing refers to it any more, which
is soon enough.

Last, the engine lets go of the file, and this cell removes the scratch folder with the database in
it:


In [19]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `Depends(get_session)` gives every request a session of its own and closes it when the request is
  finished, which is after the response has been built.
- The body a route takes is a model with no table, so a bad field is a 422 naming it; a table model
  as the body checks nothing at all.
- `response_model` builds the answer from the fields it declares, which is how a secret column stays
  out of an answer, and two models that nest into each other make a response that cannot be built.
- A missing row is `session.get` returning `None` and a route raising 404; a repeated name is an
  `IntegrityError` caught, rolled back and raised as 409.
- A test client needs an engine two threads can share: `StaticPool` for one connection and
  `check_same_thread=False` for `sqlite3`'s rule, with `dependency_overrides` pointing the
  application at it.


## What is next

The **A Small Service** notebook is the whole guide in one project: the models in a file, a
migration that builds the database, the routes of this notebook, and a test suite that runs against
a database of its own and catches the four mistakes that survive real projects.


---

&#8592; **Previous:** [Migrations](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/12-migrations.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [A Small Service](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/14-a-small-service.ipynb) &#8594;
